In [8]:
import sys
import os
import numpy as np
import torch
import shap

# import matplotlib
# matplotlib.use('tkagg') 
import matplotlib.pyplot as plt
import logging
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

# 设置项目路径
current_dir = "/home/lizihao/Work/enzyme_prediction/src"
project_root_dir = os.path.dirname(current_dir)
sys.path.insert(0, project_root_dir)

from models.model import ImprovedEnzymePredictionModel

# 设置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def evaluate_with_shap(model, test_data, device='cuda:1', output_dir='output/evaluation'):
    """评估模型并计算SHAP值"""
    os.makedirs(output_dir, exist_ok=True)
    
    model.eval()
    model = model.to(device)
    
    # 准备测试数据
    binding_site_test = torch.FloatTensor(test_data['binding_site_features']).to(device)
    substrate_test = torch.FloatTensor(test_data['substrate_features']).to(device)
    y_test = test_data['y']
    # 在加载数据后添加这些检查代码
    print("数据维度检查:")
    print(f"binding_site_features shape: {test_data['binding_site_features'].shape}")
    print(f"substrate_features shape: {test_data['substrate_features'].shape}")
    print(f"y shape: {test_data['y'].shape}")

    # 检查数据是否包含NaN或无穷值
    print("\n数据有效性检查:")
    print(f"binding_site NaN: {np.isnan(test_data['binding_site_features']).any()}")
    print(f"substrate NaN: {np.isnan(test_data['substrate_features']).any()}")
    print(f"y NaN: {np.isnan(test_data['y']).any()}")

    # 检查数值范围
    print("\n数值范围检查:")
    print(f"binding_site range: [{test_data['binding_site_features'].min()}, {test_data['binding_site_features'].max()}]")
    print(f"substrate range: [{test_data['substrate_features'].min()}, {test_data['substrate_features'].max()}]")
    print(f"y range: [{test_data['y'].min()}, {test_data['y'].max()}]")


args = type('Args', (), {
    'model_path': "/home/lizihao/Work/enzyme_prediction/output/training_20250310_205850/checkpoints/best_model.pth",
    'data_path': "/home/lizihao/Work/enzyme_prediction/output/processed/processed_data_20250310_205850.npz",
    'device': 'cuda:1',
    'output_dir': 'output/evaluation'
})()

# 加载数据
data = np.load(args.data_path)
test_data = {
    'binding_site_features': data['binding_site_features'][data['test_indices']],
    'substrate_features': data['substrate_embeddings'][data['test_indices']],
    'y': data['y'][data['test_indices']]
}
logging.info(f"Loading data from: {args.data_path}")

# 加载模型
model = ImprovedEnzymePredictionModel(binding_site_dim=7, substrate_dim=384)
checkpoint = torch.load(args.model_path, map_location=args.device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# 评估并计算SHAP值
metrics, predictions = evaluate_with_shap(model, test_data, device=args.device, output_dir=args.output_dir)

2025-04-06 21:47:02,846 - INFO - Loading data from: /home/lizihao/Work/enzyme_prediction/output/processed/processed_data_20250310_205850.npz


数据维度检查:
binding_site_features shape: (602, 100, 7)
substrate_features shape: (602, 384)
y shape: (602, 2)

数据有效性检查:
binding_site NaN: False
substrate NaN: False
y NaN: False

数值范围检查:
binding_site range: [-158.39500427246094, 2538.39990234375]
substrate range: [-2.2518224716186523, 2.2790610790252686]
y range: [-7.3872161432802645, 5.341873982291153]


TypeError: cannot unpack non-iterable NoneType object

In [ ]:
import sys
import os
import numpy as np
import torch
import shap

# import matplotlib
# matplotlib.use('tkagg') 
import matplotlib.pyplot as plt
import logging
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr

# 设置项目路径
current_dir = "/home/lizihao/Work/enzyme_prediction/src"
project_root_dir = os.path.dirname(current_dir)
sys.path.insert(0, project_root_dir)

from models.model import ImprovedEnzymePredictionModel

# 设置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def evaluate_with_shap(model, test_data, device='cuda:1', output_dir='output/evaluation'):
    """评估模型并计算SHAP值"""
    os.makedirs(output_dir, exist_ok=True)
    
    model.eval()
    model = model.to(device)
    
    # 准备测试数据
    binding_site_test = torch.FloatTensor(test_data['binding_site_features']).to(device)
    substrate_test = torch.FloatTensor(test_data['substrate_features']).to(device)
    y_test = test_data['y']
    # 在加载数据后添加这些检查代码
    print("数据维度检查:")
    print(f"binding_site_features shape: {test_data['binding_site_features'].shape}")
    print(f"substrate_features shape: {test_data['substrate_features'].shape}")
    print(f"y shape: {test_data['y'].shape}")

    # 检查数据是否包含NaN或无穷值
    print("\n数据有效性检查:")
    print(f"binding_site NaN: {np.isnan(test_data['binding_site_features']).any()}")
    print(f"substrate NaN: {np.isnan(test_data['substrate_features']).any()}")
    print(f"y NaN: {np.isnan(test_data['y']).any()}")

    # 检查数值范围
    print("\n数值范围检查:")
    print(f"binding_site range: [{test_data['binding_site_features'].min()}, {test_data['binding_site_features'].max()}]")
    print(f"substrate range: [{test_data['substrate_features'].min()}, {test_data['substrate_features'].max()}]")
    print(f"y range: [{test_data['y'].min()}, {test_data['y'].max()}]")
    # 转换为numpy数组，用于SHAP计算
    binding_site_np = binding_site_test.cpu().numpy()
    substrate_np = substrate_test.cpu().numpy()
    
    # 展平特征，便于SHAP处理
    binding_site_flat = binding_site_np.reshape(binding_site_np.shape[0], -1)  # [n_samples, 100*7]
    substrate_flat = substrate_np  # [n_samples, 384]
    X_test = np.hstack([binding_site_flat, substrate_flat])  # [n_samples, 100*7 + 384]
    
    # 创建特征名称
    feature_names = []
    for i in range(100):  # 100个残基
        for j in range(7):  # 7个特征
            feature_names.append(f"binding_site_{i}_{j}")
    for i in range(substrate_flat.shape[1]):
        feature_names.append(f"substrate_{i}")
    
    # 定义模型预测函数（SHAP需要numpy输入输出）
    def model_predict(X):
        binding_site = X[:, :100*7].reshape(-1, 100, 7)
        substrate = X[:, 100*7:]
        
        binding_site_tensor = torch.FloatTensor(binding_site).to(device)
        substrate_tensor = torch.FloatTensor(substrate).to(device)
        
        with torch.no_grad():
            predictions = model(binding_site_tensor, substrate_tensor).cpu().numpy()
        return predictions
    
    # 选择少量样本进行SHAP计算（SHAP计算成本较高）
    X_test_sample = X_test[:100]  # 选择前100个样本
    y_test_sample = y_test[:100]
    
     print("\n开始计算SHAP值...")
    explainer = shap.KernelExplainer(model_predict, X_test_sample)
    shap_values = explainer.shap_values(X_test_sample, nsamples=100)

    # 检查和转换SHAP值维度
    print(f"\nSHAP values shapes before processing:")
    for i, sv in enumerate(shap_values):
        print(f"Output {i}: {sv.shape}")
    
    # 转换SHAP值维度
    shap_values_km = shap_values[0].T if shap_values[0].shape[1] == 2 else shap_values[0]
    shap_values_kcat = shap_values[1].T if shap_values[1].shape[1] == 2 else shap_values[1]
    
    print(f"\nSHAP values shapes after processing:")
    print(f"Km SHAP values: {shap_values_km.shape}")
    print(f"kcat SHAP values: {shap_values_kcat.shape}")
    print(f"Input features: {X_test_sample.shape}")
    
    # 可视化：Summary Plot for Km
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values_km, 
        X_test_sample,
        feature_names=feature_names,
        plot_type="bar", 
        max_display=20,
        show=False
    )
    plt.title("Top 20 Feature Importance for Km Prediction")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "shap_summary_km.png"), bbox_inches='tight')
    plt.close()
    
    # 取消注释并修复kcat的可视化
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values_kcat, 
        X_test_sample,
        feature_names=feature_names,
        plot_type="bar", 
        max_display=20,
        show=False
    )
    plt.title("Top 20 Feature Importance for kcat Prediction")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "shap_summary_kcat.png"), bbox_inches='tight')
    plt.close()
    
 
    
    # 常规评估指标
    predictions = model_predict(X_test)
    metrics = {
        'mse_km': mean_squared_error(y_test[:, 0], predictions[:, 0]),
        'mse_kcat': mean_squared_error(y_test[:, 1], predictions[:, 1]),
        'r2_km': r2_score(y_test[:, 0], predictions[:, 0]),
        'r2_kcat': r2_score(y_test[:, 1], predictions[:, 1]),
        'pearson_km': pearsonr(y_test[:, 0], predictions[:, 0])[0],
        'pearson_kcat': pearsonr(y_test[:, 1], predictions[:, 1])[0]
    }
    
    logging.info("\n评估结果:")
    logging.info(f"R² - Km: {metrics['r2_km']:.4f}, kcat: {metrics['r2_kcat']:.4f}")
    logging.info(f"Pearson r - Km: {metrics['pearson_km']:.4f}, kcat: {metrics['pearson_kcat']:.4f}")
    
    return metrics, predictions


args = type('Args', (), {
    'model_path': "/home/lizihao/Work/enzyme_prediction/output/training_20250310_205850/checkpoints/best_model.pth",
    'data_path': "/home/lizihao/Work/enzyme_prediction/output/processed/processed_data_20250310_205850.npz",
    'device': 'cuda:1',
    'output_dir': 'output/evaluation'
})()

# 加载数据
data = np.load(args.data_path)
test_data = {
    'binding_site_features': data['binding_site_features'][data['test_indices']],
    'substrate_features': data['substrate_embeddings'][data['test_indices']],
    'y': data['y'][data['test_indices']]
}
logging.info(f"Loading data from: {args.data_path}")

# 加载模型
model = ImprovedEnzymePredictionModel(binding_site_dim=7, substrate_dim=384)
checkpoint = torch.load(args.model_path, map_location=args.device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# 评估并计算SHAP值
metrics, predictions = evaluate_with_shap(model, test_data, device=args.device, output_dir=args.output_dir)

2025-04-06 21:53:12,113 - INFO - Loading data from: /home/lizihao/Work/enzyme_prediction/output/processed/processed_data_20250310_205850.npz



处理后的特征维度:
Combined features shape: (602, 1084)
Expected feature dimension: 1084

开始计算SHAP值...


  0%|          | 0/50 [00:00<?, ?it/s]

2025-04-06 21:53:12,231 - INFO - num_full_subsets = 0
2025-04-06 21:53:12,234 - INFO - remaining_weight_vector = array([0.13596119, 0.06805767, 0.04542328, 0.03410617, 0.02731598,
       0.02278924, 0.01955591, 0.01713096, 0.01524492, 0.01373613,
       0.01250169, 0.01147303, 0.01060264, 0.00985663, 0.00921011,
       0.00864442, 0.00814531, 0.00770168, 0.00730476, 0.00694755,
       0.00662439, 0.00633061, 0.0060624 , 0.00581656, 0.0055904 ,
       0.00538164, 0.00518837, 0.00500892, 0.00484185, 0.00468594,
       0.0045401 , 0.00440338, 0.00427496, 0.00415411, 0.00404017,
       0.00393258, 0.00383081, 0.00373441, 0.00364296, 0.00355609,
       0.00347348, 0.0033948 , 0.00331979, 0.00324821, 0.00317981,
       0.0031144 , 0.00305177, 0.00299177, 0.00293422, 0.00287899,
       0.00282592, 0.00277491, 0.00272583, 0.00267858, 0.00263305,
       0.00258915, 0.0025468 , 0.00250592, 0.00246644, 0.00242827,
       0.00239137, 0.00235566, 0.00232109, 0.00228761, 0.00225517,
       0.0022237


SHAP values shapes:
Output 0: (1084, 2)
Output 1: (1084, 2)
Output 2: (1084, 2)
Output 3: (1084, 2)
Output 4: (1084, 2)
Output 5: (1084, 2)
Output 6: (1084, 2)
Output 7: (1084, 2)
Output 8: (1084, 2)
Output 9: (1084, 2)
Output 10: (1084, 2)
Output 11: (1084, 2)
Output 12: (1084, 2)
Output 13: (1084, 2)
Output 14: (1084, 2)
Output 15: (1084, 2)
Output 16: (1084, 2)
Output 17: (1084, 2)
Output 18: (1084, 2)
Output 19: (1084, 2)
Output 20: (1084, 2)
Output 21: (1084, 2)
Output 22: (1084, 2)
Output 23: (1084, 2)
Output 24: (1084, 2)
Output 25: (1084, 2)
Output 26: (1084, 2)
Output 27: (1084, 2)
Output 28: (1084, 2)
Output 29: (1084, 2)
Output 30: (1084, 2)
Output 31: (1084, 2)
Output 32: (1084, 2)
Output 33: (1084, 2)
Output 34: (1084, 2)
Output 35: (1084, 2)
Output 36: (1084, 2)
Output 37: (1084, 2)
Output 38: (1084, 2)
Output 39: (1084, 2)
Output 40: (1084, 2)
Output 41: (1084, 2)
Output 42: (1084, 2)
Output 43: (1084, 2)
Output 44: (1084, 2)
Output 45: (1084, 2)
Output 46: (1084, 2)
Ou

ValueError: SHAP值维度 (2) 与特征维度 (1084) 不匹配